# Unidad 3 · Clase 2 — Optimización y **despliegue** de modelos de Deep Learning

**Módulo:** Científico de Datos e IA Aplicada · **Caso:** el clasificador de prendas de la tienda de DevSenior Code (Fashion-MNIST).

> 🎯 **Hoy no entrenamos nada.** El modelo ya está entrenado, podado y cuantizado (viene en `modelos/`).
> Nuestro trabajo es el que hace un ingeniero de IA el lunes por la mañana: **medirlo, empaquetarlo como servicio y dejarlo corriendo**.

```
modelo.h5 ─► medir ─► TFLite ─► API (FastAPI) ─► Docker ─► servidor (SSH) ─► benchmark
```

**Al terminar sabrás:**
1. Medir tamaño, latencia y accuracy de un modelo antes de decidir cómo desplegarlo.
2. Explicar qué hicieron pruning y quantization, y verificar que funcionaron.
3. Servir un modelo con **FastAPI** (carga única, validación, `/health`).
4. Separar configuración y secretos con **`.env`**.
5. Empaquetar el servicio en **Docker** y comparar dos despliegues con **docker compose**.
6. Entender el rol de **SSH** para llegar a un servidor.

## 0. Preparación

In [ ]:
import os
# Los .h5 se guardaron con Keras "legacy" (tf_keras). Debe fijarse ANTES de importar tensorflow.
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import json, shutil, subprocess, sys, tempfile, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import tensorflow as tf

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# Carpeta raíz de la clase (donde viven modelos/, datos/ y despliegue/)
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "modelos").exists())
MODELOS = ROOT / "modelos"

CLASES = ["Camiseta/top", "Pantalón", "Suéter", "Vestido", "Abrigo",
          "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]

# Muestra de 1 000 imágenes REALES de test (no hace falta internet)
d = np.load(ROOT / "datos" / "fashion_mnist_muestra.npz")
X, y = d["x"], d["y"]                      # X: (1000, 28, 28) uint8 en [0, 255]
Xn = (X / 255.0).astype("float32")[..., None]   # normalizado, con canal: (1000, 28, 28, 1)
print("TensorFlow", tf.__version__, "| muestra:", X.shape, "| carpeta:", ROOT.name)

## 1. Del notebook a producción

Un modelo "que funciona en el notebook" todavía no es un producto. En producción cambian las reglas:

| | Notebook | Producción |
|---|---|---|
| Hardware | GPU, sin prisa | CPU compartida, **milisegundos** |
| Tamaño | irrelevante | arranque, memoria y costo de la imagen |
| Latencia | no se mide | es un **SLA** (p. ej. p95 < 200 ms) |
| Entorno | "en mi máquina funciona" | **idéntico en cualquier lado** (Docker) |
| Secretos | pegados en el código | fuera del código (`.env`) |

Recibimos tres artefactos ya producidos por el equipo de modelado. **Primer reflejo del ingeniero de despliegue: mirar qué recibió.**

In [ ]:
# ✍️ TU TURNO
# Lista los archivos de la carpeta MODELOS y muestra el tamaño de cada uno en KB.
# Pista: MODELOS.iterdir() recorre la carpeta; f.stat().st_size da los bytes.

**Lectura rápida:** `modelo_base.h5` es el original; `modelo_podado.h5` tiene pesos poda­dos (ya pesa menos al comprimirse); `modelo_optimizado.tflite` es podado **+ cuantizado + formato de inferencia**.

## 2. Medir antes de decidir

La regla de oro: **nunca se optimiza ni se despliega a ciegas**. Medimos tres cosas y las comparamos:

| Métrica | Pregunta |
|---|---|
| **Tamaño** | ¿cuánto pesa el artefacto? |
| **Accuracy** | ¿perdimos calidad al optimizar? |
| **Latencia** | ¿cuántos ms tarda UNA predicción? |

Primero, funciones de evaluación. La de Keras viene lista; **la de TFLite la escribimos nosotros** (es exactamente lo que hará la API).

In [ ]:
def evaluar_keras(ruta, n_latencia=200):
    """Accuracy (lote completo) y latencia por imagen (una a una) de un modelo Keras .h5."""
    modelo = tf.keras.models.load_model(ruta)
    pred = modelo.predict(Xn, verbose=0).argmax(axis=1)
    acc = float((pred == y).mean())
    modelo.predict(Xn[:1], verbose=0)                       # calentamiento
    t = time.perf_counter()
    for i in range(n_latencia):
        modelo.predict(Xn[i:i+1], verbose=0)
    lat = (time.perf_counter() - t) / n_latencia * 1000
    return {"accuracy": acc, "latencia_ms": lat}

In [ ]:
# ✍️ TU TURNO
def evaluar_tflite(ruta, n_latencia=200):
    # 1) Crea un tf.lite.Interpreter(model_path=str(ruta)) y llama allocate_tensors()
    # 2) Obtén los índices con get_input_details()[0]["index"] y get_output_details()[0]["index"]
    # 3) Para predecir: set_tensor(entrada, img[None, ...]) -> invoke() -> get_tensor(salida)[0].argmax()
    # 4) Calcula accuracy sobre Xn, y la latencia promedio (ms) de n_latencia llamadas una a una
    # Devuelve {"accuracy": ..., "latencia_ms": ...}
    pass

In [ ]:
# ✍️ TU TURNO
# Evalúa los 3 artefactos y arma un DataFrame con: tamaño_KB, accuracy, latencia_ms.
# Usa evaluar_keras para los .h5 y evaluar_tflite para el .tflite.
# Agrega dos columnas: cuántas veces más chico (x_mas_chico) y más rápido (x_mas_rapido) que el base.

In [ ]:
# ✍️ TU TURNO
# Dibuja 3 gráficos de barras (tamaño, latencia, accuracy) comparando los 3 modelos.
# Pista: plt.subplots(1, 3) y a.bar(nombres, df[columna])

### 🧠 Interpretar (no solo calcular)
- ¿Perdimos accuracy? Con 1 000 imágenes, diferencias de ±0.5 pts **no son significativas**: es ruido de muestreo.
- La latencia baja de forma espectacular (órdenes de magnitud), pero ojo: comparamos `model.predict()` (pensado para lotes y con mucho *overhead*) contra el intérprete TFLite (pensado para una predicción). Parte de la ganancia es **la herramienta correcta para servir**, no solo la cuantización.
- La decisión: **desplegamos el `.tflite`**.

## 3. ¿Qué hicieron *pruning* y *quantization*? (sin reentrenar)

El equipo de modelado aplicó dos técnicas. No las repetimos, pero un buen ingeniero **verifica que hicieron lo que dicen**.

| Técnica | Idea | Qué esperamos ver |
|---|---|---|
| **Pruning** | poner en cero los pesos que casi no aportan | muchos pesos exactamente en 0 |
| **Quantization** | guardar pesos en int8 en vez de float32 | ~4× menos tamaño |
| *Distillation* | un modelo pequeño imita a uno grande | (solo concepto; requiere reentrenar) |

In [ ]:
# ✍️ TU TURNO
# Escribe dispersion(ruta): carga el modelo, junta TODOS los pesos (m.get_weights()) en un solo vector
# y devuelve la fracción que vale exactamente 0, y el total de pesos.
# Llámala para modelo_base.h5 y modelo_podado.h5 e imprime el resultado.

**Conversión a TFLite** — esto sí es parte del despliegue (es rápido y no reentrena). Probemos dos tipos de cuantización sobre el modelo podado y comparemos tamaños:

In [ ]:
# ✍️ TU TURNO
# Convierte modelo_podado.h5 a TFLite de tres formas y compara el tamaño en KB:
#   1) sin cuantizar   2) dinámica (Optimize.DEFAULT)   3) float16 (+ target_spec.supported_types=[tf.float16])
# Pista: tf.lite.TFLiteConverter.from_keras_model(modelo) -> .optimizations -> .convert()

| Tipo | Pesos | Tamaño típico | Cuándo |
|---|---|---|---|
| Dinámica (int8) | int8 | ~4× menos | punto de partida, sin datos de calibración |
| Float16 | float16 | ~2× menos | si el hardware lo soporta (GPU/edge) |
| Entera completa (int8) | int8 + activaciones | ~4× y más rápido en CPU/edge | móviles, microcontroladores; **requiere dataset de calibración** |
| QAT | int8 simulado al entrenar | ~4× con menos pérdida | cuando la post-training degrada demasiado |

**Formatos de despliegue:** `SavedModel` (TF Serving) · `.h5` (legado, no para producción nueva) · **`.tflite`** (liviano, móviles/microservicios) · **`.onnx`** (portable entre frameworks, ONNX Runtime/TensorRT).

## 4. Servir el modelo: una API con FastAPI

Un modelo solo sirve si otros sistemas pueden **pedirle** predicciones. Lo exponemos por HTTP:

```
cliente ──POST /predict {imagen}──► [ FastAPI ► valida ► TFLite ] ──► {clase, confianza, latencia}
          ◄──────── JSON ─────────
```

Buenas prácticas que aplicamos:
1. **Cargar el modelo una sola vez** al arrancar (jamás en cada petición).
2. **Validar** la entrada con Pydantic (errores claros, no silenciosos).
3. Separar `model_utils.py` (inferencia) de `main.py` (endpoints).
4. Exponer **`/health`** para que Docker/Kubernetes sepan si el servicio vive.

Armamos una API de trabajo en `api_trabajo/`. Los esquemas y las utilidades vienen listos; **el `main.py` lo escribimos juntos.**

In [ ]:
API_DIR = ROOT / "api_trabajo"
(API_DIR / "app").mkdir(parents=True, exist_ok=True)
(API_DIR / "model").mkdir(exist_ok=True)
shutil.copy(MODELOS / "modelo_optimizado.tflite", API_DIR / "model" / "modelo_optimizado.tflite")

# Reutilizamos los módulos ya probados del proyecto de despliegue
BASE_PROY = ROOT / "despliegue" / "01_api_fastapi_docker" / "app"
for nombre in ["__init__.py", "schemas.py", "model_utils.py"]:
    shutil.copy(BASE_PROY / nombre, API_DIR / "app" / nombre)

print((API_DIR / "app" / "schemas.py").read_text()[:900])

In [ ]:
# ✍️ TU TURNO
# Escribe app/main.py. Necesitas:
#   - un lifespan (asynccontextmanager) que llame obtener_modelo() UNA vez al arrancar
#   - GET  /health  -> HealthResponse(status="ok", modelo_cargado=..., version_modelo=...)
#   - POST /predict -> recibe PredictionRequest, llama _modelo.predecir(np.array(request.imagen))
#                      y devuelve PredictionResponse; 503 si no hay modelo, 422 si la imagen es inválida
# Guarda el texto con (API_DIR / "app" / "main.py").write_text(codigo_main)
# (El proyecto completo está en despliegue/01_api_fastapi_docker/app/main.py si te trabas)

Levantamos el servidor **desde el notebook** (proceso aparte) y le hablamos como lo haría cualquier cliente:

In [ ]:
def levantar_api(puerto=8010, extra_env=None):
    env = {**os.environ, **(extra_env or {})}
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--port", str(puerto)],
        cwd=API_DIR, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):                               # espera a que /health responda
        try:
            if requests.get(f"http://127.0.0.1:{puerto}/health", timeout=1).ok:
                return proc
        except requests.exceptions.RequestException:
            time.sleep(1)
    proc.kill(); raise RuntimeError("La API no arrancó")

api = levantar_api()
URL = "http://127.0.0.1:8010"
print(requests.get(f"{URL}/health").json())

In [ ]:
# ✍️ TU TURNO
# Envía la imagen X[0] al endpoint POST /predict (JSON: {"imagen": <lista 28x28>}) y muestra la respuesta.
# Pista: requests.post(f"{URL}/predict", json={...}); compara con CLASES[y[0]]

In [ ]:
# ✍️ TU TURNO
# Prueba dos entradas inválidas y observa el código HTTP: (1) imagen con forma incorrecta, (2) JSON sin el campo "imagen".
# ¿Qué código devuelve FastAPI? ¿Quién lo detectó: Pydantic o nuestro código?

In [ ]:
# ✍️ TU TURNO
# Mide 200 peticiones: compara la latencia que reporta la API ("latencia_ms") con el tiempo total que ves como cliente.
# ¿Por qué no son iguales? Anota en un comentario dónde se va el resto del tiempo.

**Idea central:** la latencia que ve el usuario = *inferencia* + *serialización JSON* + *red* + *framework*. Optimizar el modelo ataca solo la primera parte; en producción hay que medir la **latencia extremo a extremo**.

Abre ahora `http://127.0.0.1:8010/docs`: FastAPI generó una documentación interactiva sin escribir una línea extra.

In [ ]:
api.terminate(); api.wait()
print("API detenida")

## 5. Configuración y secretos: archivos `.env`

Un `.env` guarda la configuración **que cambia entre entornos o que es secreta** (rutas, niveles de log, API keys). El código no debe tener esos valores pegados.

| Sin `.env` | Con `.env` |
|---|---|
| Claves en el código → quedan en el historial de git | Los secretos viven fuera del repositorio |
| Un `main.py` distinto por entorno | El mismo código sirve en dev, staging y producción |
| Cambiar un valor = redesplegar código | Cambiar el `.env` y reiniciar |

**Regla de oro:** el `.env` real **nunca** se versiona (va en `.gitignore` y `.dockerignore`); se versiona un `.env.example` sin valores sensibles.

Nuestra API ya lee `LOG_LEVEL` y `API_KEY` del entorno. Si `API_KEY` existe, `/predict` exige la cabecera `X-API-Key`.

In [ ]:
# ✍️ TU TURNO
# 1) Escribe un archivo .env en API_DIR con LOG_LEVEL=DEBUG y API_KEY=clave-de-clase-2
# 2) Escribe leer_env(ruta) que devuelva un diccionario {CLAVE: valor} (ignora líneas vacías y comentarios)
# 3) Imprime la config OCULTANDO el valor de cualquier clave que contenga "KEY"
#    (en proyectos reales se usa python-dotenv: from dotenv import load_dotenv)

In [ ]:
# ✍️ TU TURNO
# Levanta la API en el puerto 8011 pasando la config del .env (extra_env=config).
# Envía /predict (a) sin cabecera y (b) con la cabecera X-API-Key. ¿Qué códigos esperas? Luego detén la API.

## 6. Empaquetar con Docker

> *"Funciona en mi máquina"* deja de ser excusa: la **imagen** lleva el sistema, las dependencias, el código y el modelo.

| Concepto | Qué es | Analogía |
|---|---|---|
| **Dockerfile** | receta de construcción | la receta impresa |
| **Imagen** | plantilla inmutable | el plato "empaquetado" |
| **Contenedor** | imagen en ejecución | el plato servido |
| **Registry** | donde se guardan imágenes | el recetario compartido |

Este es el Dockerfile del proyecto (léelo, cada línea tiene una razón):

In [ ]:
print((ROOT / "despliegue" / "01_api_fastapi_docker" / "Dockerfile").read_text())

**Por qué está escrito así**
- `python:3.11-slim`: base pequeña → menos superficie de ataque y builds más rápidos.
- `COPY requirements.txt` **antes** que el código: si solo cambias código, Docker reutiliza la capa de `pip install` (caché de capas).
- Solo se copian `app/` y `model/`: **ni entrenamiento, ni datos, ni `.env`** (por eso el `.dockerignore`).
- `HEALTHCHECK` usa nuestro `/health`: Docker marca el contenedor como *healthy* solo cuando el modelo cargó.
- Versiones fijas en `requirements.txt`: la imagen de hoy = la de dentro de seis meses.

### 💻 Ahora en la **terminal** (no en el notebook)
```bash
cd despliegue/01_api_fastapi_docker

# 1. Construir la imagen
docker build -t clase02-inference-api .

# 2. Ejecutar el contenedor pasando la configuración por entorno (sin copiar el .env a la imagen)
cp .env.example .env            # edita API_KEY
docker run -d --name api --env-file .env -p 8000:8000 clase02-inference-api

# 3. Verificar
docker ps                        # estado "healthy" tras unos segundos
python test_api.py --api-key LA_CLAVE_QUE_PUSISTE
docker logs api                  # los logs de la aplicación

# 4. Limpiar
docker stop api && docker rm api
```

In [ ]:
# ¿Tienes Docker disponible en esta máquina?
if shutil.which("docker"):
    print(subprocess.run(["docker", "--version"], capture_output=True, text=True).stdout.strip())
else:
    print("Docker no está instalado/en el PATH. Instálalo (Docker Desktop) o sigue la demo del docente.")

### 🧩 Reflexión: ¿cuánto pesa la imagen?
Corre `docker images clase02-inference-api`. Verás **más de 1 GB**: el modelo pesa 110 KB, ¡pero `tensorflow-cpu` pesa cientos de MB! Para servir un `.tflite` solo necesitamos el *runtime*, no todo TensorFlow.
**Reto:** reemplazar `tensorflow-cpu` por `ai-edge-litert` (mismo `Interpreter`) y comparar el tamaño final de la imagen. Esa es la verdadera optimización de despliegue.

## 7. Dos despliegues, un comando: `docker compose`

Para comparar **base vs optimizado** en condiciones reales levantamos los dos servicios a la vez:

| Servicio | Modelo | Puerto |
|---|---|---|
| `modelo-base` | `modelo_base.h5` con Keras | 8001 |
| `modelo-optimizado` | `modelo_optimizado.tflite` | 8002 |

```bash
cd despliegue/02_benchmark_compose
docker compose up --build -d            # construye y levanta ambos
docker compose ps                        # ambos "healthy"

pip install -r benchmark/requirements.txt
python benchmark/compare.py --n 200      # mismas imágenes reales a los dos; genera benchmark/reporte.md

docker compose down                      # apagar y limpiar
```

El script mide **tamaño, latencia (media y p95), throughput y accuracy** con las mismas 200 imágenes.

In [ ]:
reporte = ROOT / "despliegue" / "02_benchmark_compose" / "benchmark" / "reporte.md"
print(reporte.read_text() if reporte.exists() else "Aún no hay reporte: corre benchmark/compare.py en la terminal y vuelve a ejecutar esta celda.")

### Cómo leer el resultado
- **p95** importa más que el promedio: es lo que vive el 5 % de usuarios más desafortunado (los SLA se definen sobre percentiles).
- **Throughput** = peticiones/segundo que aguanta *este* contenedor. Si necesitas más, se escala **horizontalmente** (más réplicas detrás de un balanceador).
- Si el accuracy es igual y la latencia baja, la optimización fue gratis; si no, es un trade-off que se discute con negocio.

## 8. ¿Cómo llega todo esto a un servidor? SSH

Construimos la imagen en nuestra máquina. Para ejecutarla en un servidor (EC2, VM de GCP/Azure, un VPS, un servidor de la empresa) nos conectamos por **SSH**: un canal cifrado donde **la llave privada nunca viaja**.

```
Tu máquina                         Servidor
[ llave PRIVADA ] ◄─ desafío ─►  [ llave PÚBLICA en ~/.ssh/authorized_keys ]
```
*La pública es un candado que le entregas al servidor; la privada es la única llave que lo abre.*

```bash
ssh-keygen -t ed25519 -C "tu_correo@ejemplo.com"     # una sola vez
ssh-copy-id usuario@servidor                         # instala tu llave pública
ssh usuario@servidor                                 # entrar sin contraseña
scp modelo_optimizado.tflite usuario@servidor:/srv/modelos/
ssh usuario@servidor "docker compose up -d"          # desplegar sin sesión interactiva
```

In [ ]:
# ✍️ TU TURNO
# Genera un par de llaves ed25519 SIN passphrase en una carpeta temporal (solo para la demo) usando subprocess + ssh-keygen.
# Muestra la llave pública y SOLO la primera línea de la privada. Borra la carpeta al final.
# Pista: ["ssh-keygen", "-t", "ed25519", "-N", "", "-f", ruta, "-q"]

**Buenas prácticas:** autenticar siempre con **llaves** (nunca contraseña) · proteger la privada con *passphrase* · no subirla jamás a git (GitHub escanea y alerta de fugas) · usar `~/.ssh/config`:
```
Host servidor-produccion
    HostName 203.0.113.10
    User despliegue
    IdentityFile ~/.ssh/id_ed25519
```

## 9. Checklist de despliegue

- [ ] **Medí** tamaño, accuracy y latencia (y la latencia *extremo a extremo*)
- [ ] El modelo se **carga una sola vez**, al arrancar
- [ ] Entrada **validada** (Pydantic) y errores con códigos HTTP claros
- [ ] Endpoint **`/health`** y `HEALTHCHECK` en el Dockerfile
- [ ] Configuración y secretos en **`.env`** (fuera de git y fuera de la imagen)
- [ ] Imagen **slim**, dependencias **fijadas**, solo lo necesario dentro
- [ ] Probé con **datos reales**, no con ruido aleatorio
- [ ] Sé cómo **llegar al servidor** (SSH con llaves) y cómo hacer *rollback* (imagen anterior)

## 🏁 Retos (tarea)
1. Cambia a **float16** en la conversión y compara tamaño/latencia con el dinámico.
2. Añade un endpoint **`/predict_batch`** que acepte varias imágenes.
3. Reemplaza `tensorflow-cpu` por **`ai-edge-litert`** en la imagen y compara el tamaño final.
4. Haz un **multi-stage build** del Dockerfile.
5. Agrega un tercer servicio al compose con el modelo **podado sin cuantizar** y compara los tres.
6. Exporta a **ONNX** (`tf2onnx`) y sirve con `onnxruntime`.

## 🔭 Puente a la siguiente clase
Si esta API recibiera 10 000 peticiones por segundo, ¿qué cambiaría? ¿Cómo versionarías el modelo en producción? ¿Y si el **entrenamiento** también viviera en una plataforma como **Databricks**?

In [ ]:
# Limpieza: elimina la carpeta de trabajo creada durante la clase
shutil.rmtree(ROOT / "api_trabajo", ignore_errors=True)
print("Listo ✅")